In [13]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

In [14]:
import sklearn; print(f'Sklearn version: {sklearn.__version__}')

Sklearn version: 1.3.2


In [15]:
diabetes = load_diabetes(scaled=False)
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='target')

In [16]:
print(f"Размер данных: {X.shape}")
print(f"Признаки: {list(X.columns)}")
print(f"Первые 5 строк:")
X.head()

Размер данных: (442, 10)
Признаки: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Первые 5 строк:


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0


In [17]:
# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (353, 10)
Размер тестовой выборки: (89, 10)


In [19]:
# Создаем пайплайн с масштабированием и случайным лесом
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

In [20]:
# Запускаем MLflow эксперимент
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("diabetes_prediction")

with mlflow.start_run(run_name="random_forest_baseline") as run:
    # Обучаем модель
    pipeline.fit(X_train, y_train)
    
    # Делаем предсказания
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    
    # Вычисляем метрики
    train_mse = mean_squared_error(y_train, y_pred_train)
    train_r2 = r2_score(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Логируем параметры
    mlflow.log_param("n_estimators", pipeline.named_steps['rf'].n_estimators)
    mlflow.log_param("max_depth", pipeline.named_steps['rf'].max_depth)
    mlflow.log_param("random_state", pipeline.named_steps['rf'].random_state)
    
    # Логируем метрики
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_r2", test_r2)
    
    # Логируем модель
    mlflow.sklearn.log_model(
        pipeline,
        "model",
        registered_model_name="diabetes_model"
    )
    
    print(f"Run ID: {run.info.run_id}")
    print(f"Experiment ID: {run.info.experiment_id}")
    print(f"Train MSE: {train_mse:.2f}")
    print(f"Test MSE: {test_mse:.2f}")
    print(f"Train R2: {train_r2:.3f}")
    print(f"Test R2: {test_r2:.3f}")

2026/03/14 14:24:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'diabetes_model' already exists. Creating a new version of this model...
2026/03/14 14:24:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes_model, version 2


Run ID: b34852f3ebed4110bfd114a99a238a53
Experiment ID: 1
Train MSE: 529.52
Test MSE: 2975.70
Train R2: 0.913
Test R2: 0.438
🏃 View run random_forest_baseline at: http://localhost:5000/#/experiments/1/runs/b34852f3ebed4110bfd114a99a238a53
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '2' of model 'diabetes_model'.


In [21]:
# Показываем важность признаков
feature_importance = pd.DataFrame({
    'feature': diabetes.feature_names,
    'importance': pipeline.named_steps['rf'].feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков:")
print(feature_importance)


Важность признаков:
  feature  importance
2     bmi    0.357678
8      s5    0.232822
3      bp    0.088534
9      s6    0.069799
0     age    0.057971
5      s2    0.057823
4      s1    0.051422
6      s3    0.051128
7      s4    0.023558
1     sex    0.009265


In [22]:
# Сохраняем модель локально для использования в сервисе
import joblib
joblib.dump(pipeline, '../models/diabetes_model.joblib')
print("Модель сохранена локально")

Модель сохранена локально


In [12]:
import sklearn; print(f'Sklearn version: {sklearn.__version__}')

Sklearn version: 1.3.2


In [23]:
# Проверяем загрузку модели
loaded_model = joblib.load('../models/diabetes_model.joblib')
sample_data = X_test.iloc[0:1]
sample_pred = loaded_model.predict(sample_data)
print(f"Пример предсказания: {sample_pred[0]:.2f}")

Пример предсказания: 144.96
